# Hindi Movie Sentiment Analysis (FastText)

**Google Colab Backup Version**

1. Before running, click the **Folder icon** on the left menu.
2. Drag and drop your dataset file: `Dataset_CCE2.xlsx - train.csv` into the Colab file browser.
3. Run the cells below one by one.

In [ ]:
!pip install fasttext pandas

In [ ]:
import pandas as pd
import fasttext
import re
import os

# 1. Load the dataset (Handles both root and sample_data path)
file_path = 'Dataset_CCE2.xlsx - train.csv'
if not os.path.exists(file_path): file_path = 'sample_data/Dataset_CCE2.xlsx - train.csv'

df = pd.read_csv(file_path)
df = df.dropna(subset=['text', 'experience'])

# 2. Clean text utility
def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'\n', ' ', text)
    return text.strip()

# 3. Convert to FastText format
with open('train.txt', 'w', encoding='utf-8') as f:
    for _, row in df.iterrows():
        try:
            label = str(int(row['experience']))
            text = clean_text(row['text'])
            f.write(f'__label__{label} {text}\n')
        except:
            continue

print(f"Successfully processed {len(df)} reviews!")

In [ ]:
# 4. Train Model!
print("Training FastText model. This takes a few seconds...")
model = fasttext.train_supervised(input='train.txt', epoch=25, wordNgrams=2)
print("Training Complete!")

In [ ]:
from IPython.display import clear_output
clear_output()
print("======================================")
print("     🎬 Hindi Movie Sentiment Chatbot    ")
print("       (Type 'exit' or 'quit' to stop)   ")
print("======================================\n")

while True:
    user_input = input("\nYou: ")
    
    if user_input.lower() in ['exit', 'quit']:
        print("\nChatbot: Bye! 👋")
        break
        
    if not user_input.strip():
        continue
        
    cleaned = clean_text(user_input)
    labels, probs = model.predict(cleaned)
    raw_label = labels[0].replace('__label__', '')
    
    sentiment_map = {
        '0': '😞 Negative (नकारात्मक)',
        '1': '😐 Neutral (तटस्थ)',
        '2': '😃 Positive (सकारात्मक)'
    }
    
    sentiment = sentiment_map.get(raw_label, 'Unknown')
    print(f"Chatbot: {sentiment} | Confidence: {probs[0]*100:.2f}%")
